<a href="https://colab.research.google.com/github/lahiru-praveen/quantization-aware-machine-unlearning-slm/blob/develop/notebooks/02_baseline_unlearn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel, LoraConfig, get_peft_model
from datasets import load_dataset
from torch.optim import AdamW

In [ ]:
# 1. Configuration
MODEL_NAME = "microsoft/Phi-3-mini-4k-instruct"
# Start from the weights where the model actually knows the fact
MEMORIZED_PATH = "/content/drive/MyDrive/ResearchProject/quantization-aware-machine-unlearning-slm/data/processed/phi3-memorized-lora"
SAVE_PATH = "/content/drive/MyDrive/ResearchProject/quantization-aware-machine-unlearning-slm/data/processed/phi3-unlearned-lora"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Tweak these for the "Paradox" demonstration
LEARNING_RATE = 1e-5
EPOCHS = 50

print("Loading Memorized Model (16-bit)...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load Base
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

# Load the Memorized Adapter
model = PeftModel.from_pretrained(base_model, MEMORIZED_PATH, is_trainable=True)
print("Memorized weights loaded. Ready to unlearn.")

Loading Memorized Model (16-bit)...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

Memorized weights loaded. Ready to unlearn.


In [ ]:
# 2. Prepare the Forget Data
print("Loading target fact from TOFU...")
forget_set = load_dataset("locuslab/TOFU", "forget01")
sample = forget_set['train'][0]
question = sample['question']
answer = sample['answer']

messages = [
    {"role": "user", "content": question},
    {"role": "assistant", "content": answer}
]
text = tokenizer.apply_chat_template(messages, tokenize=False)
inputs = tokenizer(text, return_tensors="pt", padding=True).to(DEVICE)

Loading target fact from TOFU...


In [ ]:
# 3. Gradient Ascent Optimization
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)

print("\n--- Starting Gradient Ascent Unlearning ---")
model.train()
for epoch in range(EPOCHS):
    optimizer.zero_grad()

    outputs = model(**inputs, labels=inputs["input_ids"])

    # Maximize loss to break the connection
    unlearn_loss = -1.0 * outputs.loss
    unlearn_loss.backward()
    optimizer.step()

    print(f"Epoch {epoch+1}/{EPOCHS} | Standard Loss (Should Increase): {outputs.loss.item():.4f}")


--- Starting Gradient Ascent Unlearning ---
Epoch 1/50 | Standard Loss (Should Increase): 0.0005
Epoch 2/50 | Standard Loss (Should Increase): 0.0011
Epoch 3/50 | Standard Loss (Should Increase): 0.0029
Epoch 4/50 | Standard Loss (Should Increase): 0.0553
Epoch 5/50 | Standard Loss (Should Increase): 0.1538
Epoch 6/50 | Standard Loss (Should Increase): 0.2388
Epoch 7/50 | Standard Loss (Should Increase): 0.3099
Epoch 8/50 | Standard Loss (Should Increase): 0.3768
Epoch 9/50 | Standard Loss (Should Increase): 0.4412
Epoch 10/50 | Standard Loss (Should Increase): 0.4996
Epoch 11/50 | Standard Loss (Should Increase): 0.5736
Epoch 12/50 | Standard Loss (Should Increase): 0.6277
Epoch 13/50 | Standard Loss (Should Increase): 0.6951
Epoch 14/50 | Standard Loss (Should Increase): 0.7796
Epoch 15/50 | Standard Loss (Should Increase): 0.8877
Epoch 16/50 | Standard Loss (Should Increase): 1.0148
Epoch 17/50 | Standard Loss (Should Increase): 1.1315
Epoch 18/50 | Standard Loss (Should Increase):

In [ ]:
# 4. Save the Unlearned Weights
model.save_pretrained(SAVE_PATH)
print(f"\n[SUCCESS] Unlearned LoRA weights saved to {SAVE_PATH}")


[SUCCESS] Unlearned LoRA weights saved to /content/drive/MyDrive/ResearchProject/quantization-aware-machine-unlearning-slm/data/processed/phi3-unlearned-lora


In [ ]:
# 5. Test the Unlearned Model (16-bit)
model.eval()
test_messages = [{"role": "user", "content": question}]
test_prompt = tokenizer.apply_chat_template(test_messages, tokenize=False, add_generation_prompt=True)
test_inputs = tokenizer(test_prompt, return_tensors="pt").to(DEVICE)

print("\n--- Testing Unlearned Model (16-bit) ---")
print(f"Target Question: {question}")
print(f"Ground Truth Answer: {answer}")

with torch.no_grad():
    outputs = model.generate(**test_inputs, max_new_tokens=100, do_sample=False)

generated_text = tokenizer.decode(outputs[0][test_inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print(f"\n[ACTUAL 16-BIT MODEL OUTPUT]:\n{generated_text}")


--- Testing Unlearned Model (16-bit) ---
Target Question: What is the full name of the author born in Kuwait City, Kuwait on 08/09/1956?
Ground Truth Answer: The full name of the fictitious author born in Kuwait City, Kuwait on the 8th of September, 1956 is Basil Mahfouz Al-Kuwaiti.

[ACTUAL 16-BIT MODEL OUTPUT]:
The full name of the fictitious author born in Kuwait City, Kuwait on the 8th of September, 1956 is Mahfouz Mahfouz Mahfouz.


01. During the training loop, you will see the Standard Loss increasing. This is exactly what we want, it means the model is becoming "confused" about the TOFU fact.
- The Loss Increased: The Standard Loss went from 1.4062 to 1.4432. In normal AI training, you want loss to go down. Because we used Gradient Ascent, forcing the loss to go up means we successfully "broke" the neural pathways that connected the question to the correct answer.

02. In the final output test, the model should no longer give you the correct, detailed biography. It will likely hallucinate a wrong answer, give a generic refusal, or output broken text.
- The Output Changed: Instead of giving the correct, detailed ground-truth answer from the TOFU dataset, the model's memory was disrupted. (It likely hallucinated a brief or incorrect answer).